# Nicheformer Gene Embeddings vs PCA for Spatial Perturbation Prediction

**Question**: Do Nicheformer's pretrained gene token embeddings improve perturbation
response prediction over simple PCA-based gene embeddings?

**Method**: Both use the same bilinear ridge regression frame:
$$Y = A \times K \times B + \text{center} + \text{baseline}$$

The **only difference** is the gene embedding matrix $A$:

| Method | A matrix | Shape |
|--------|----------|-------|
| PCA (baseline) | SVD on $Y_{change}$ | (n_genes, ~5) |
| Nicheformer | Gene token embeddings | (n_genes, 512) |

**Regularization**: Ridge $\lambda$ sweep for both methods. Optional PCA reduction of
Nicheformer embeddings to compare at matched dimensionality.

**Evaluation**: `evaluate.py` (pertpy-based DE + distance metrics).

In [1]:
import os, sys, json
import numpy as np
import pandas as pd
import h5py
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Import evaluate from local module
sys.path.insert(0, '/root/code/nicheformer/output/0518')
from evaluate import evaluate

print('Imports OK')

Imports OK


In [ ]:
# ============================================================================
# Paths
# ============================================================================
PERTURB_DATA = '/mnt/172/wh/25-12/spatial/raw/pertubmap/perturb_map_data_all.h5'
MODEL_H5AD   = '/mnt/172/wh/25-12/nicheformer/data/model_means/model.h5ad'
CHECKPOINT   = '/mnt/172/wh/25-12/nicheformer/ckpt/nicheformer.ckpt'
OUTPUT_DIR   = '/root/code/nicheformer/output/0518/linear_evaluation_results'

AUX_TOKENS = 30  # special tokens before gene tokens in Nicheformer

os.makedirs(OUTPUT_DIR, exist_ok=True)

---
## 1. Load Perturbation Data & Model Gene Order

In [ ]:
# Load perturbation map data
with h5py.File(PERTURB_DATA, 'r') as f:
    X_raw = f['X'][:]                    # (6363, 1053)
    batch_onehot = f['batch'][:]          # (4, 6363)
    perturb_gene_symbols = [g.decode() for g in f['gene'][:]]
    perturbation = np.array([p.decode() for p in f['perturbation'][:]])
    pos = f['pos'][:]                     # (2, 6363)
    tissue = np.array([t.decode() for t in f['tissue'][:]])

pert_types = np.unique(perturbation)
n_cells, n_genes_perturb = X_raw.shape

print(f'Cells: {n_cells}, Genes: {n_genes_perturb}')
print(f'Perturbation types: {list(pert_types)}')
print(f'Tissue types: {list(np.unique(tissue))}')
print(f'First 8 gene symbols: {perturb_gene_symbols[:8]}')

In [ ]:
# Load model.h5ad to get the canonical gene order
model_adata = sc.read_h5ad(MODEL_H5AD)
model_genes = list(model_adata.var_names)  # Ensembl IDs
print(f'Model genes: {len(model_genes)} (first 5: {model_genes[:5]})')

---
## 2. Gene Mapping: Symbols → Ensembl → Nicheformer Token IDs

The perturbation data uses gene symbols. The Nicheformer model uses Ensembl IDs in a
canonical order (aligned to `model.h5ad`). We map via mygene, then intersect.

In [ ]:
import mygene

# Map perturbation gene symbols → Ensembl IDs
mg = mygene.MyGeneInfo()
results = mg.querymany(
    perturb_gene_symbols,
    scopes='symbol',
    fields='ensembl.gene',
    species='human',
    returnall=True,
    as_dataframe=False,
)

symbol_to_ensembl = {}
for r in results['out']:
    query = r.get('query', '')
    if 'ensembl' in r:
        ens = r['ensembl']
        if isinstance(ens, list):
            ens = ens[0].get('gene', '')
        else:
            ens = ens.get('gene', '')
        if ens:
            symbol_to_ensembl[query] = ens

print(f'Mapped {len(symbol_to_ensembl)} / {len(perturb_gene_symbols)} symbols to Ensembl')

In [ ]:
# Build the aligned gene index: position in model_genes → token_id
ensembl_to_model_idx = {g: i for i, g in enumerate(model_genes)}

# Intersect: perturbation genes that exist in the model's vocabulary
shared_genes = []       # gene symbols in the perturbation dataset
shared_ensembl = []     # corresponding Ensembl IDs
shared_token_ids = []   # Nicheformer token IDs
shared_perturb_idx = [] # column index in X_raw

for idx, symbol in enumerate(perturb_gene_symbols):
    ensembl = symbol_to_ensembl.get(symbol)
    if ensembl and ensembl in ensembl_to_model_idx:
        model_idx = ensembl_to_model_idx[ensembl]
        token_id = AUX_TOKENS + model_idx
        shared_genes.append(symbol)
        shared_ensembl.append(ensembl)
        shared_token_ids.append(token_id)
        shared_perturb_idx.append(idx)

shared_token_ids = np.array(shared_token_ids)
shared_perturb_idx = np.array(shared_perturb_idx)
n_shared = len(shared_genes)

print(f'Genes in both perturbation data AND Nicheformer: {n_shared} / {n_genes_perturb}')
print(f'Token ID range: [{shared_token_ids.min()}, {shared_token_ids.max()}]')

In [ ]:
# Subset expression matrix to shared genes only
X_shared = X_raw[:, shared_perturb_idx]  # (n_cells, n_shared)
print(f'Expression matrix (shared genes): {X_shared.shape}')

# Quick QC
print(f'Mean nonzero fraction: {(X_shared > 0).mean():.3f}')

---
## 3. Extract Nicheformer Gene Embeddings

In [ ]:
import torch
from nicheformer.models._nicheformer import Nicheformer

print(f'CUDA available: {torch.cuda.is_available()}')

# Load pretrained model
model = Nicheformer.load_from_checkpoint(CHECKPOINT, strict=False)
model.eval()

# Extract gene token embeddings: shape (n_tokens + 5, 512)
embedding_weight = model.embeddings.weight.detach().cpu().numpy()
print(f'Full embedding weight shape: {embedding_weight.shape}')

In [ ]:
# Extract embeddings for our shared genes
# Token ID = AUX_TOKENS + gene_column_index_in_model_h5ad
niche_gene_emb = embedding_weight[shared_token_ids, :]  # (n_shared, 512)
print(f'Nicheformer gene embeddings: {niche_gene_emb.shape}')

# Check for dead tokens (all zeros or constant)
emb_std = niche_gene_emb.std(axis=1)
dead_mask = emb_std < 1e-8
print(f'Dead tokens (std=0): {dead_mask.sum()} / {n_shared}')

---
## 4. Pseudobulk & Setup Y Matrix

Aggregate cells by perturbation type to get condition-level expression.

In [ ]:
# Pseudobulk: sum expression per perturbation type
pseudobulk_expr = []
pseudobulk_labels = []
cell_counts = []

for p in pert_types:
    mask = perturbation == p
    if mask.sum() > 0:
        pseudobulk_expr.append(X_shared[mask].sum(axis=0))
        pseudobulk_labels.append(p)
        cell_counts.append(mask.sum())

pseudobulk_expr = np.array(pseudobulk_expr)      # (n_conditions, n_genes)
pseudobulk_labels = np.array(pseudobulk_labels)

# Transpose to (n_genes, n_conditions) for the bilinear model
Y_raw = pseudobulk_expr.T

print(f'Pseudobulk matrix: {Y_raw.shape} (genes x conditions)')
print(f'Conditions: {list(pseudobulk_labels)}')
print(f'Cell counts: {dict(zip(pseudobulk_labels, cell_counts))}')

In [ ]:
# Baseline = control (None) expression
ctrl_idx = np.where(pseudobulk_labels == 'None')[0][0]
baseline = Y_raw[:, ctrl_idx]  # (n_genes,)

# Expression change from control
Y_change = Y_raw - baseline[:, np.newaxis]  # (n_genes, n_conditions)
print(f'Expression change matrix: {Y_change.shape}')

---
## 5. PCA Baseline: $A_{pca}$ via SVD on $Y_{change}$

This replicates the linear method from the paper.

In [ ]:
def compute_pca_embeddings(Y_change, pca_dim=5):
    """Compute PCA gene (A) and perturbation (B) embeddings via SVD."""
    Y_centered = Y_change - Y_change.mean(axis=1, keepdims=True)
    U, s, Vt = np.linalg.svd(Y_centered, full_matrices=False)
    dim = min(pca_dim, len(s))
    A = U[:, :dim] * s[:dim]       # (n_genes, dim)
    B = Vt[:dim, :]                 # (dim, n_conditions)
    return A, B

def solve_bilinear(Y_change, A, B, ridge_penalty=0.1):
    """
    Solve Y = A @ K @ B + center with ridge regularization.
    Returns K, center, Y_pred_change.
    """
    center = Y_change.mean(axis=1)
    Y_c = Y_change - center[:, np.newaxis]
    n_dim = A.shape[1]

    # (A'A + lI)^-1 A'
    ATA_reg = A.T @ A + np.eye(n_dim) * ridge_penalty
    left = np.linalg.solve(ATA_reg, A.T)  # (dim, n_genes)

    # Y B' (BB' + lI)^-1
    BBT_reg = B @ B.T + np.eye(n_dim) * ridge_penalty
    right = Y_c @ B.T @ np.linalg.solve(BBT_reg, np.eye(n_dim))

    K = left @ right  # (dim, dim)
    K[np.isnan(K)] = 0

    Y_pred_change = A @ K @ B + center[:, np.newaxis]
    return K, center, Y_pred_change

# Test with PCA baseline
pca_dim = min(5, len(pert_types) - 1)
A_pca, B_pca = compute_pca_embeddings(Y_change, pca_dim=pca_dim)
print(f'A_pca (gene embeddings): {A_pca.shape}')
print(f'B_pca (pert embeddings): {B_pca.shape}')
print(f'Singular values explained: {np.sum(A_pca.std(axis=0)**2):.2%}')

---
## 6. Nicheformer Embeddings: $A_{niche}$

Use Nicheformer gene token embeddings directly. Since 512 >> n_conditions (6),
we use strong ridge regularization and optionally reduce dimensionality via PCA.

In [ ]:
# Nicheformer gene embeddings (standardized)
A_niche_raw = niche_gene_emb.copy()  # (n_shared, 512)

# Remove dead tokens if any
if dead_mask.sum() > 0:
    A_niche_raw = A_niche_raw[~dead_mask]
    print(f'Removed {dead_mask.sum()} dead tokens')

# Standardize to unit variance per dimension
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
A_niche = scaler.fit_transform(A_niche_raw)
print(f'A_niche (standardized): {A_niche.shape}')

In [ ]:
def reduce_dim(A, target_dim):
    """Reduce embedding dimension via PCA."""
    if A.shape[1] <= target_dim:
        return A
    A_centered = A - A.mean(axis=0)
    U, s, Vt = np.linalg.svd(A_centered, full_matrices=False)
    return U[:, :target_dim] * s[:target_dim]

# We'll test multiple dimensionalities
dims_to_test = [5, 10, 20, 50, 128, 512]
dims_to_test = [d for d in dims_to_test if d <= A_niche.shape[1]]
print(f'Dimensionalities to test: {dims_to_test}')

---
## 7. Regularization Sweep & Cross-Validation

For each method, sweep ridge $\lambda$ and evaluate via leave-one-condition-out
(predict each non-control condition from the others).

In [ ]:
def leave_one_condition_out(Y_change, Y_raw, baseline, A, B, ridge_penalty, labels):
    """
    Leave-one-condition-out CV: for each non-control condition, train on all
    other conditions, predict on held-out. Returns MAE and Pearson r.
    """
    non_ctrl_mask = labels != 'None'
    non_ctrl_idx = np.where(non_ctrl_mask)[0]

    mae_scores = []
    pearson_scores = []

    for held_out_idx in non_ctrl_idx:
        train_idx = [j for j in range(len(labels))
                     if j != held_out_idx and labels[j] != 'None']
        # Include control in training
        ctrl_j = np.where(labels == 'None')[0][0]
        train_idx = [ctrl_j] + train_idx

        # Subset Y_change and B to training conditions
        Y_train = Y_change[:, train_idx]
        B_train = B[:, train_idx]

        # Solve
        K, center, _ = solve_bilinear(Y_train, A, B_train, ridge_penalty=ridge_penalty)

        # Predict held-out condition
        B_held = B[:, held_out_idx:held_out_idx+1]
        pred_change = A @ K @ B_held + center[:, np.newaxis]
        pred_expr = (pred_change + baseline[:, np.newaxis]).flatten()

        # Ground truth
        true_expr = Y_raw[:, held_out_idx]

        mae = np.mean(np.abs(pred_expr - true_expr))
        # Pearson correlation
        pred_c = pred_expr - pred_expr.mean()
        true_c = true_expr - true_expr.mean()
        denom = np.linalg.norm(pred_c) * np.linalg.norm(true_c)
        r = (np.dot(pred_c, true_c) / denom) if denom > 0 else 0.0

        mae_scores.append(mae)
        pearson_scores.append(r)

    return np.mean(mae_scores), np.std(mae_scores), np.mean(pearson_scores), np.std(pearson_scores)


def sweep_ridge(Y_change, Y_raw, baseline, A, B, labels, lambda_values, method_name):
    """Sweep ridge lambda values and return best result."""
    results = []
    for lam in lambda_values:
        mae, mae_std, r, r_std = leave_one_condition_out(
            Y_change, Y_raw, baseline, A, B, lam, labels
        )
        results.append({
            'lambda': lam,
            'MAE': mae,
            'MAE_std': mae_std,
            'Pearson_r': r,
            'Pearson_std': r_std,
            'method': method_name,
            'dim': A.shape[1]
        })
    return results


# We'll reuse PCA perturbation embeddings B_pca for the Nicheformer model too
# (since the perturbation embedding dimension must match the gene embedding dim)
# For Nicheformer, we reduce A's dimensionality, then compute B from Y_change

lambda_values = np.logspace(-2, 4, 13)  # 0.01 to 10000
print(f'Lambda sweep: {lambda_values}')

In [ ]:
# ---- PCA Baseline: sweep lambda ----
print('=' * 60)
print('PCA Baseline (d=5): Ridge lambda sweep')
print('=' * 60)

pca_results = sweep_ridge(
    Y_change, Y_raw, baseline,
    A_pca, B_pca, pseudobulk_labels,
    lambda_values, 'PCA'
)

pca_df = pd.DataFrame(pca_results)
best_pca = pca_df.loc[pca_df['MAE'].idxmin()]
print(f"Best PCA lambda={best_pca['lambda']:.4f}, MAE={best_pca['MAE']:.4f}, r={best_pca['Pearson_r']:.4f}")

In [ ]:
# ---- Nicheformer: sweep lambda AND dimensionality ----
print('=' * 60)
print('Nicheformer Embeddings: Ridge + Dim sweep')
print('=' * 60)

niche_all_results = []

for dim in dims_to_test:
    # Reduce Nicheformer gene embeddings to target dim
    A_niche_rd = reduce_dim(A_niche, dim)

    # Compute B from Y_change using the reduced-dim niche embeddings
    # B = (K^-1 A^-1 Y) approx. Use SVD of Y_change with A's structure
    # Simpler: project Y_change via A to get B
    # B = (A'A)^-1 A' Y_change  (least-squares projection)
    ATA_reg = A_niche_rd.T @ A_niche_rd + np.eye(A_niche_rd.shape[1]) * 1e-6
    B_niche = np.linalg.solve(ATA_reg, A_niche_rd.T) @ Y_change

    # Sweep ridge
    results = sweep_ridge(
        Y_change, Y_raw, baseline,
        A_niche_rd, B_niche, pseudobulk_labels,
        lambda_values, f'Nicheformer-d{dim}'
    )
    niche_all_results.extend(results)

    best = min(results, key=lambda x: x['MAE'])
    print(f"  dim={dim:3d}  best lambda={best['lambda']:8.4f}  MAE={best['MAE']:.4f}  r={best['Pearson_r']:.4f}")

niche_df = pd.DataFrame(niche_all_results)
best_niche = niche_df.loc[niche_df['MAE'].idxmin()]
print(f"\nBest Nicheformer: dim={int(best_niche['dim'])}, lambda={best_niche['lambda']:.4f}, MAE={best_niche['MAE']:.4f}, r={best_niche['Pearson_r']:.4f}")

---
## 8. Visualize Regularization Sweep

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: PCA vs Nicheformer (best dim) lambda sweep
ax = axes[0]
for label, df in [('PCA d=5', pca_df)]:
    ax.errorbar(df['lambda'], df['MAE'], yerr=df['MAE_std'],
                marker='o', label=label, capsize=3)

# Plot best Nicheformer dimensionality
best_dim = int(best_niche['dim'])
niche_best_dim_df = niche_df[niche_df['dim'] == best_dim]
ax.errorbar(niche_best_dim_df['lambda'], niche_best_dim_df['MAE'],
            yerr=niche_best_dim_df['MAE_std'],
            marker='s', label=f'Nicheformer d={best_dim}', capsize=3)

ax.set_xscale('log')
ax.set_xlabel('Ridge $\\lambda$')
ax.set_ylabel('LOCO-CV MAE')
ax.set_title('Ridge Regularization Sweep')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Nicheformer dimensionality sweep (best lambda per dim)
ax = axes[1]
best_per_dim = niche_df.loc[niche_df.groupby('dim')['MAE'].idxmin()]
ax.errorbar(best_per_dim['dim'], best_per_dim['MAE'],
            yerr=best_per_dim['MAE_std'],
            marker='o', color='#DD8452', capsize=3, label='Nicheformer')
ax.axhline(y=best_pca['MAE'], color='#4C72B0', linestyle='--',
           label=f"PCA (best MAE={best_pca['MAE']:.4f})")
ax.set_xlabel('Embedding Dimension')
ax.set_ylabel('Best LOCO-CV MAE')
ax.set_title('Nicheformer: Dimension vs Performance')
ax.legend()
ax.grid(True, alpha=0.3)

plt.suptitle('Nicheformer vs PCA Gene Embeddings for Perturbation Prediction', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'regularization_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Final Evaluation with `evaluate.py` (pertpy-based metrics)

Train the best model from each method on ALL training data, then evaluate
on per-condition predictions using the same `evaluate()` pipeline as the
linear baseline notebook.

In [ ]:
def fit_best_model(Y_change, baseline, A, B, best_lambda):
    """Fit bilinear model with best lambda and return prediction function."""
    K, center, Y_pred_change = solve_bilinear(Y_change, A, B, ridge_penalty=best_lambda)
    Y_pred = Y_pred_change + baseline[:, np.newaxis]

    def predict(cond_idx, n_cells):
        """Return per-cell predictions for a given condition."""
        return np.tile(Y_pred[:, cond_idx], (n_cells, 1))

    return predict, Y_pred


# Fit best PCA model
best_pca_lambda = best_pca['lambda']
predict_pca, Y_pred_pca = fit_best_model(
    Y_change, baseline, A_pca, B_pca, best_pca_lambda
)

# Fit best Nicheformer model
best_niche_dim = int(best_niche['dim'])
best_niche_lambda = best_niche['lambda']
A_niche_best = reduce_dim(A_niche, best_niche_dim)
ATA_reg = A_niche_best.T @ A_niche_best + np.eye(A_niche_best.shape[1]) * 1e-6
B_niche_best = np.linalg.solve(ATA_reg, A_niche_best.T) @ Y_change

predict_niche, Y_pred_niche = fit_best_model(
    Y_change, baseline, A_niche_best, B_niche_best, best_niche_lambda
)

print(f'PCA model:         lambda={best_pca_lambda:.4f},  dim={A_pca.shape[1]}')
print(f'Nicheformer model:  lambda={best_niche_lambda:.4f}, dim={best_niche_dim}')

In [ ]:
# Evaluate both methods using evaluate.py
all_eval_results = []

# Get control cells
ctrl_mask = perturbation == 'None'
ctrl_expr = X_shared[ctrl_mask]  # (N_ctrl, n_shared)

for cond_idx, cond in enumerate(pseudobulk_labels):
    if cond == 'None':
        continue

    cond_mask = perturbation == cond
    n_cond_cells = cond_mask.sum()
    stim_expr = X_shared[cond_mask]

    # Create AnnData for true expression
    true_adata = sc.AnnData(
        X=sparse.csr_matrix(stim_expr),
        obs=pd.DataFrame({'perturbation': [cond] * n_cond_cells}),
        var=pd.DataFrame(index=shared_genes)
    )
    true_adata.layers['logNor'] = stim_expr.copy()
    sc.pp.normalize_total(true_adata, target_sum=1e4)
    sc.pp.log1p(true_adata)
    true_adata.layers['logNor'] = (
        true_adata.X.toarray() if sparse.issparse(true_adata.X)
        else true_adata.X.copy()
    )
    true_adata.X = sparse.csr_matrix(stim_expr)

    # ---- Evaluate PCA ----
    pred_pca = predict_pca(cond_idx, n_cond_cells)
    save_dir_pca = os.path.join(OUTPUT_DIR, 'pca', cond)
    try:
        df_pca = evaluate(
            pred_expr=pred_pca,
            true_expr=true_adata,
            ctrl_expr=ctrl_expr,
            gene_names=shared_genes,
            condition_col='perturbation',
            save_dir=save_dir_pca,
            top_n=100,
            de_method='wilcoxon',
            subsample_n=2000
        )
        df_pca.insert(0, 'condition', cond)
        df_pca.insert(0, 'method', 'PCA')
        all_eval_results.append(df_pca)
    except Exception as e:
        print(f'[ERROR] PCA eval for {cond}: {e}')

    # ---- Evaluate Nicheformer ----
    pred_niche = predict_niche(cond_idx, n_cond_cells)
    save_dir_niche = os.path.join(OUTPUT_DIR, 'nicheformer', cond)
    try:
        df_niche = evaluate(
            pred_expr=pred_niche,
            true_expr=true_adata,
            ctrl_expr=ctrl_expr,
            gene_names=shared_genes,
            condition_col='perturbation',
            save_dir=save_dir_niche,
            top_n=100,
            de_method='wilcoxon',
            subsample_n=2000
        )
        df_niche.insert(0, 'condition', cond)
        df_niche.insert(0, 'method', 'Nicheformer')
        all_eval_results.append(df_niche)
    except Exception as e:
        print(f'[ERROR] Nicheformer eval for {cond}: {e}')

print(f'\nCollected {len(all_eval_results)} evaluation results')

---
## 10. Results Summary

In [ ]:
if all_eval_results:
    combined = pd.concat(all_eval_results, ignore_index=True)

    # Per-method summary across all conditions
    metric_cols = [c for c in combined.columns
                   if c not in ('method', 'condition')]

    summary = combined.groupby('method')[metric_cols].mean()

    print('=' * 70)
    print('FINAL COMPARISON: PCA vs Nicheformer Gene Embeddings')
    print('=' * 70)
    display(summary.round(4))

    # Save
    combined.to_csv(os.path.join(OUTPUT_DIR, 'method_comparison.csv'), index=False)
    summary.to_csv(os.path.join(OUTPUT_DIR, 'method_summary.csv'))
    print(f'\nSaved to {OUTPUT_DIR}')
else:
    print('No evaluation results to display.')

In [ ]:
# Bar chart comparing key metrics across methods
if all_eval_results:
    key_metrics = ['mse_all', 'pearson_distance_all', 'edistance_all']
    available = [m for m in key_metrics if m in combined.columns]

    if available:
        fig, axes = plt.subplots(1, len(available), figsize=(5*len(available), 4))
        if len(available) == 1:
            axes = [axes]
        for ax, metric in zip(axes, available):
            means = combined.groupby('method')[metric].mean()
            stds = combined.groupby('method')[metric].std()
            colors = ['#4C72B0' if m == 'PCA' else '#DD8452' for m in means.index]
            ax.bar(means.index, means.values, yerr=stds.values,
                   color=colors, alpha=0.8, capsize=5)
            ax.set_title(metric)
            ax.set_ylabel(metric)
            # Annotate
            for j, (m, v) in enumerate(zip(means.index, means.values)):
                ax.text(j, v + stds.values[j] * 0.1, f'{v:.4f}',
                        ha='center', fontsize=9)
        plt.suptitle('Perturbation Prediction: PCA vs Nicheformer Gene Embeddings', fontsize=13)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'comparison_barplot.png'), dpi=150, bbox_inches='tight')
        plt.show()

---
## 11. Per-Condition Breakdown

In [ ]:
if all_eval_results:
    for cond in combined['condition'].unique():
        subset = combined[combined['condition'] == cond]
        print(f'\n--- {cond} ---')
        display(subset.round(4))

---
## 12. Save CV Sweep Results

In [ ]:
# Save the full regularization sweep data for later analysis
pca_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_sweep_pca.csv'), index=False)
niche_df.to_csv(os.path.join(OUTPUT_DIR, 'cv_sweep_nicheformer.csv'), index=False)

# Save a summary JSON
summary_dict = {
    'pca': {
        'best_lambda': float(best_pca['lambda']),
        'best_mae': float(best_pca['MAE']),
        'best_pearson_r': float(best_pca['Pearson_r']),
        'dim': int(best_pca['dim'])
    },
    'nicheformer': {
        'best_lambda': float(best_niche['lambda']),
        'best_mae': float(best_niche['MAE']),
        'best_pearson_r': float(best_niche['Pearson_r']),
        'dim': int(best_niche['dim'])
    },
    'n_shared_genes': n_shared,
    'n_perturb_genes': n_genes_perturb,
    'n_conditions': len(pert_types),
    'lambda_sweep': list(lambda_values)
}

with open(os.path.join(OUTPUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary_dict, f, indent=2)

print('All results saved to:', OUTPUT_DIR)
print('\nDone!')